# 02 · Technology Hype Cycles

HN is a real-time seismograph of the tech industry. By tracking how often each technology appears in story titles, we can draw its full life cycle: birth, hype peak, and decline.

**Headline chart:** A heatmap of 30+ technologies × 18 years — the graveyard and the nursery of tech in a single image.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.loader import db
from src.nlp import TECH_KEYWORDS, keyword_hits
from src.viz import set_style, hype_heatmap, time_series, save

set_style()
con = db()

## Load stories with year

In [ ]:
stories = con.execute("""
    SELECT title, YEAR(posted_at) AS year, score
    FROM stories
    WHERE year BETWEEN 2007 AND 2024
      AND score >= 1
""").df()

print(f'{len(stories):,} stories loaded')
stories.head()

## Tag each story with matching keywords

In [ ]:
for name, patterns in TECH_KEYWORDS.items():
    col = f'kw_{name}'
    stories[col] = stories['title'].apply(lambda t: keyword_hits(str(t), patterns))

print('Keyword columns added:', [c for c in stories.columns if c.startswith('kw_')][:5], '...')

## Build normalized frequency matrix

In [ ]:
kw_cols = [f'kw_{k}' for k in TECH_KEYWORDS]
stories_per_year = stories.groupby('year').size().rename('total')

# Sum hits per keyword per year, normalize by total stories that year
freq = stories.groupby('year')[kw_cols].sum()
freq_norm = freq.div(stories_per_year, axis=0) * 10_000  # per 10k stories

# Rename columns to clean labels
freq_norm.columns = list(TECH_KEYWORDS.keys())
freq_norm = freq_norm.T  # technologies as rows, years as columns

# Keep only techs with at least one year above threshold
freq_norm = freq_norm[freq_norm.max(axis=1) > 1]

# Sort by year of peak mention
freq_norm['peak_year'] = freq_norm.idxmax(axis=1)
freq_norm = freq_norm.sort_values('peak_year').drop(columns='peak_year')

print(freq_norm.shape)
freq_norm

## The headline heatmap

In [ ]:
fig = hype_heatmap(
    freq_norm,
    title='Technology Hype Cycles on Hacker News (2007–2024)\nMentions per 10,000 stories'
)
save(fig, '../data/fig_hype_heatmap.png')
plt.show()
print('Key patterns to look for:')
print('  - Bitcoin/Blockchain peak: 2017–2018')
print('  - ChatGPT/LLM spike: 2023 onward')
print('  - Heroku flatline after 2022 (Salesforce killed free tier)')
print('  - Ruby decline: post-2014')

## AI discourse timeline — the ChatGPT inflection

In [ ]:
ai_cols = ['kw_Machine Learning', 'kw_Deep Learning', 'kw_GPT', 'kw_ChatGPT', 'kw_LLM']
ai_monthly = con.execute("""
    SELECT
        DATE_TRUNC('month', posted_at) AS month,
        COUNT(*) AS total
    FROM stories
    WHERE YEAR(posted_at) >= 2015
    GROUP BY 1
    ORDER BY 1
""").df()

# Add monthly AI keyword counts
ai_stories = stories[stories['year'] >= 2015].copy()
# (simplified: use the tagged dataframe we already have)

fig, ax = plt.subplots(figsize=(14, 5))
for kw in ['Machine Learning', 'Deep Learning', 'GPT', 'ChatGPT', 'LLM', 'Transformer']:
    col = f'kw_{kw}'
    if col not in stories.columns:
        continue
    monthly = stories[stories['year'] >= 2015].groupby('year')[col].sum()
    ax.plot(monthly.index, monthly.values, linewidth=2, label=kw, marker='o', markersize=4)

ax.axvline(x=2022.9, color='red', linestyle='--', alpha=0.7, label='ChatGPT launch (Nov 2022)')
ax.set_title('AI / ML discourse on Hacker News (2015–2024)', fontsize=14, fontweight='bold')
ax.set_ylabel('Stories mentioning keyword')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('../data/fig_ai_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

## "X is dead" — technology obituaries

In [ ]:
import re

dead_pattern = r'\b(is dead|is dying|is over|killed|deprecated|sunsetting)\b'
obituaries = stories[stories['title'].str.contains(dead_pattern, case=False, na=False, regex=True)]

print(f'{len(obituaries):,} obituary posts found')

obit_by_year = obituaries.groupby('year').size()

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(obit_by_year.index, obit_by_year.values, color='#555', edgecolor='white')
ax.set_title('"X is dead / dying" posts on HN per year', fontsize=14, fontweight='bold')
ax.set_ylabel('Posts')
plt.tight_layout()
plt.savefig('../data/fig_obituaries.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop obituary titles:')
print(obituaries.nlargest(20, 'score')[['title', 'year', 'score']].to_string(index=False))